# Spark Standalone – Ejercicios prácticos y Proyecto (Análisis de Stocks)

Notebook con la resolución de los **7 ejercicios**, el **proyecto de análisis de stocks** (6 pasos) y las **6 preguntas de reflexión**.
Pensado para ejecutarse en **Google Colab**, Jupyter local o el entorno BDA Lab (Docker). Ejecuta las celdas en orden.

## 0. Preparación del entorno

In [1]:
# En Colab pyspark suele no venir instalado. Si ya lo tienes, esta celda no hace nada.
try:
    import pyspark
    print("pyspark", pyspark.__version__)
except ImportError:
    !pip install -q pyspark
    import pyspark
    print("pyspark instalado:", pyspark.__version__)

pyspark 4.0.5


In [2]:
import os, shutil, time, urllib.request
from pyspark.sql import SparkSession

BASE_URL = "https://github.com/josepgarcia/datos/raw/refs/heads/main/"
os.makedirs("DATOS", exist_ok=True)
os.makedirs("RESULTADOS", exist_ok=True)

def descargar(nombre):
    """Descarga un archivo del repo de datos a ./DATOS. Devuelve True si existe al final."""
    destino = os.path.join("DATOS", nombre)
    if not os.path.exists(destino):
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except Exception as e:
            print(f"No se pudo descargar {nombre}: {e}")
            return False
    return True

def ruta_local(path_relativo):
    """Ruta absoluta con prefijo file:// para que Spark lea del disco local y no de HDFS."""
    return "file://" + os.path.abspath(path_relativo)

---
## Ejercicio 1: Configurar SparkSession personalizada

In [3]:
# Si ya hay una sesión activa, getOrCreate() la reutilizaría ignorando la nueva configuración.
SparkSession.builder.getOrCreate().stop()

spark = (SparkSession.builder
    .appName("MiPrimerAnalisis")                        # 1. nombre
    .config("spark.executor.memory", "1g")              # 2. 1 GB por executor
    .config("spark.sql.shuffle.partitions", "8")        # 3. 8 particiones para shuffle
    .getOrCreate())
sc = spark.sparkContext

# 4. Verificar que se creó correctamente
print("Versión de Spark :", spark.version)
print("Nombre de la app :", sc.appName)
print("Master           :", sc.master)
print("executor.memory  :", sc.getConf().get("spark.executor.memory"))
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
spark

Versión de Spark : 4.0.5
Nombre de la app : MiPrimerAnalisis
Master           : local[*]
executor.memory  : 1g
shuffle.partitions: 8


**Resultado:** la SparkSession se crea con el nombre `MiPrimerAnalisis`, `spark.executor.memory = 1g` y `spark.sql.shuffle.partitions = 8`.
En modo `local[*]` no existen executors separados (todo corre dentro del driver), por lo que la memoria de executor queda configurada pero solo tiene efecto real en un cluster (Standalone/YARN).

---
## Ejercicio 2: Operaciones básicas con RDDs

In [4]:
numeros = list(range(1, 21))          # [1, 2, ..., 20]

rdd = sc.parallelize(numeros)         # 1. crear RDD

total   = rdd.count()                 # 2. número de elementos
primeros = rdd.take(5)                # 3. primeros 5
suma    = rdd.sum()                   # 4. suma
maximo, minimo = rdd.max(), rdd.min() # 5. máximo y mínimo

print("Total de elementos:", total)
print("Primeros 5:", primeros)
print("Suma:", suma)
print(f"Máximo: {maximo}, Mínimo: {minimo}")

assert (total, primeros, suma, maximo, minimo) == (20, [1,2,3,4,5], 210, 20, 1)

Total de elementos: 20
Primeros 5: [1, 2, 3, 4, 5]
Suma: 210
Máximo: 20, Mínimo: 1


---
## Ejercicio 3: Transformaciones `map` y `filter`

In [5]:
numeros = sc.parallelize(range(1, 51))                  # 1 a 50

por_tres  = numeros.map(lambda x: x * 3)                # 1. multiplicar por 3   (transformación, lazy)
mayores50 = por_tres.filter(lambda x: x > 50)           # 2. solo > 50           (transformación, lazy)
cuantos   = mayores50.count()                           # 3. contar              (ACCIÓN: aquí se ejecuta todo)
primeros10 = mayores50.take(10)                         # 4. primeros 10

print("Después de multiplicar por 3:", por_tres.take(5), "... hasta", por_tres.max())
print("Mayores a 50:", mayores50.take(3), "... hasta", mayores50.max())
print("Total de números mayores a 50:", cuantos)
print("Primeros 10:", primeros10)

assert cuantos == 34
assert primeros10 == [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]

Después de multiplicar por 3: [3, 6, 9, 12, 15] ... hasta 150
Mayores a 50: [51, 54, 57] ... hasta 150
Total de números mayores a 50: 34
Primeros 10: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


---
## Ejercicio 4: Pares clave-valor

In [6]:
ventas = [
    ("Laptop", 1200),
    ("Mouse", 25),
    ("Laptop", 1100),
    ("Teclado", 75),
    ("Mouse", 30),
    ("Laptop", 1300),
]

rdd_ventas = sc.parallelize(ventas)                                    # 1. RDD de pares

total_por_producto = rdd_ventas.reduceByKey(lambda a, b: a + b)        # 2. total por producto
num_ventas = (rdd_ventas.map(lambda kv: (kv[0], 1))                    # 3. nº de ventas por producto
                        .reduceByKey(lambda a, b: a + b))
mejor = total_por_producto.sortBy(lambda kv: kv[1], ascending=False).first()   # 4. mayor total
# (alternativa: total_por_producto.reduce(lambda a, b: a if a[1] >= b[1] else b))

print("Total por producto:", dict(total_por_producto.collect()))
print("Número de ventas  :", dict(num_ventas.collect()))
print(f"Producto con mayor total: {mejor[0]} ({mejor[1]})")

assert dict(total_por_producto.collect()) == {"Laptop": 3600, "Mouse": 55, "Teclado": 75}
assert dict(num_ventas.collect()) == {"Laptop": 3, "Mouse": 2, "Teclado": 1}
assert mejor == ("Laptop", 3600)

Total por producto: {'Laptop': 3600, 'Teclado': 75, 'Mouse': 55}
Número de ventas  : {'Laptop': 3, 'Teclado': 1, 'Mouse': 2}
Producto con mayor total: Laptop (3600)


---
## Ejercicio 5: Lectura y procesamiento de un CSV (`persons.csv`)

In [7]:
# 1. Descargar persons.csv (si no hay internet, se crea con el contenido de ejemplo del enunciado)
if not descargar("persons.csv"):
    with open("DATOS/persons.csv", "w", encoding="utf-8") as f:
        f.write("name,age,city\nAna,28,Madrid\nJuan,35,Barcelona\nMaría,42,Valencia\n")
    print("Se creó persons.csv de ejemplo (3 filas).")

# 2. Cargar con textFile usando ruta ABSOLUTA con file://
rdd = sc.textFile(ruta_local("DATOS/persons.csv"))
print("Primeras líneas:", rdd.take(3))

# 3. Eliminar el encabezado
header = rdd.first()
sin_header = rdd.filter(lambda linea: linea != header)

# 4. Extraer nombre y edad (split UNA sola vez por línea)
def nombre_edad(linea):
    cols = linea.split(",")
    return (cols[0], int(cols[1]))

personas = sin_header.map(nombre_edad)

# 5. Filtrar mayores de 30
mayores_30 = personas.filter(lambda p: p[1] > 30)

# 6. Contar
print("Personas mayores de 30:", mayores_30.collect())
print("Total:", mayores_30.count(), "personas")

Primeras líneas: ['name,age,gender,salary', 'John Doe,30,Male,50000', 'Jane Smith,25,Female,45000']
Personas mayores de 30: [('David Johnson', 35), ('Michael Wilson', 40), ('Sarah Brown', 32), ('James Martinez', 38), ('William Anderson', 33), ('Karen Hernandez', 31), ('Christopher Taylor', 37), ('Matthew Davis', 36), ('Daniel Miller', 34), ('Linda Martin', 39)]
Total: 10 personas


---
## Ejercicio 6: Optimización – evitar múltiples `split()`

**1. Problema de eficiencia.** El código original define tres transformaciones independientes (`nombres`, `edades`, `ciudades`) y cada una hace `x.split(",")` sobre **cada línea**. Si se consumen las tres, el `split()` se ejecuta 3 veces por línea y además el archivo se lee y recorre 3 veces (al ser lazy y no estar cacheado el RDD, cada acción vuelve a leer el archivo y recalcular).

In [8]:
# ❌ Versión ineficiente (para comparar)
rdd = sc.textFile(ruta_local("DATOS/persons.csv"))
nombres_i  = rdd.map(lambda x: x.split(",")[0])
edades_i   = rdd.map(lambda x: x.split(",")[1])
ciudades_i = rdd.map(lambda x: x.split(",")[2])

# ✅ 2 y 3. Versión optimizada: split UNA vez y extraer los campos en una sola pasada
# (funciona con cualquier nº de columnas: el persons.csv real puede ser name,age,gender,salary)
header = rdd.first()

def parsear(linea):
    return tuple(linea.split(","))                 # un único split por línea

personas_tupla = rdd.filter(lambda l: l != header).map(parsear).cache()   # cache: no se relee ni se re-parsea

nombres  = personas_tupla.map(lambda t: t[0])      # estas operaciones ya no hacen split
edades   = personas_tupla.map(lambda t: t[1])
ciudades = personas_tupla.map(lambda t: t[2])      # 3ª columna (city en el enunciado; en tu archivo puede ser gender)

print(personas_tupla.take(4))

[('John Doe', '30', 'Male', '50000'), ('Jane Smith', '25', 'Female', '45000'), ('David Johnson', '35', 'Male', '60000'), ('Emily Davis', '28', 'Female', '52000')]


**4. Por qué es más eficiente:**
- `split()` pasa de ejecutarse **3 veces por línea a 1**. Con 1 millón de líneas son 3 millones de splits frente a 1 millón (≈ 2/3 menos de trabajo de parseo).
- Con `.cache()` el archivo se lee y se parsea una sola vez; las siguientes acciones usan los datos ya en memoria en vez de volver a leer del disco.
- Se genera menos objetos temporales (listas de strings) → menos presión sobre el recolector de basura.
- En Spark el coste real de un `map` por registro se multiplica por el número de registros y de particiones, así que ahorrar una operación por línea escala linealmente con el tamaño del dataset.

---
## Ejercicio 7: Debugging – encuentra y corrige errores

| # | Error | Por qué es un error |
|---|-------|---------------------|
| 1 | Falta obtener el `SparkContext` (`sc`) | Se usa `sc.textFile(...)` pero `sc` nunca se definió: solo se creó `spark`. Hay que hacer `sc = spark.sparkContext` (o usar `spark.sparkContext.textFile`). |
| 2 | Ruta incorrecta para archivo local | `"./datos.csv"` sin prefijo se interpreta con el sistema de archivos por defecto (en BDA Lab, HDFS) y relativo al working dir del executor → `FileNotFoundException`. Hay que usar ruta **absoluta** con `file://`. |
| 3 | `collect()` sobre un dataset grande | Trae **todos** los datos al driver; si no caben en su memoria → `OutOfMemoryError` / `Py4JJavaError`. Usar `take(n)` para una muestra o `foreach()` para procesar en los executors. |
| 4 | No se cierra la sesión | El puerto 4040 de la Spark UI y los recursos (memoria/CPU) quedan ocupados; en Jupyter impide crear otra sesión. Falta `spark.stop()`. |

In [9]:
# ✅ Código corregido
import os
from pyspark.sql import SparkSession

# (Error 1) Sesión con configuración explícita y SparkContext
spark = (SparkSession.builder
    .appName("Ejercicio7")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
sc = spark.sparkContext                                   # <- faltaba obtener el SparkContext

# (Error 2) Ruta absoluta con file://
# (aquí se reutiliza persons.csv porque datos.csv es un nombre de ejemplo)
path = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd = sc.textFile(path)

# (Error 3) NO usar collect() en datasets grandes
for linea in rdd.take(100):                               # muestra acotada
    print(linea)
# o, para recorrer todo sin traerlo al driver:
# rdd.foreach(lambda l: None)   # se ejecuta en los executors (el print saldría en sus logs)

# (Error 4) Cerrar la sesión
spark.stop()

name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Michael Wilson,40,Male,75000
Sarah Brown,32,Female,58000
Robert Lee,29,Male,51000
Lisa Garcia,27,Female,49000
James Martinez,38,Male,70000
Jennifer Rodriguez,26,Female,47000
William Anderson,33,Male,62000
Karen Hernandez,31,Female,55000
Christopher Taylor,37,Male,69000
Mary Gonzalez,24,Female,44000
Matthew Davis,36,Male,67000
Patricia White,29,Female,50000
Daniel Miller,34,Male,64000
Elizabeth Jackson,30,Female,52000
Joseph Harris,28,Male,53000
Linda Martin,39,Female,71000


---
# Proyecto: Análisis de `stocks.txt`

> **Nota importante sobre los datos.** El enunciado habla de "logs de servidor web" y describe un `stocks.txt` de bolsa (`Símbolo,Fecha,Precio,Volumen`), pero el archivo real del repositorio es un **inventario de productos** con columnas `id,name,category,quantity,price` (~30 filas). Para que el notebook funcione con el archivo real, el Paso 2 detecta el formato y lo convierte a un esquema común `(grupo, etiqueta, precio, cantidad)`:
>
> | Esquema común | Datos de bolsa (enunciado) | Inventario (archivo real) |
> |---|---|---|
> | grupo | símbolo | categoría |
> | etiqueta | fecha | producto |
> | precio | precio | precio |
> | cantidad | volumen | cantidad en stock |
>
> Así los Pasos 3–6 son los mismos del enunciado y sirven con cualquiera de los dos formatos.

## Paso 1: Configuración inicial y carga de datos

In [10]:
from pyspark.sql import SparkSession
import os

# Reiniciar sesión por si quedó una anterior (ej. del ejercicio 7)
SparkSession.builder.getOrCreate().stop()

# 1-2. SparkSession "AnalisisStocks" con 512 MB de memoria para el executor
spark = (SparkSession.builder
    .appName("AnalisisStocks")
    .config("spark.executor.memory", "512m")
    .getOrCreate())

# 3. SparkContext
sc = spark.sparkContext

# Descarga del dataset
if not descargar("stocks.txt"):
    # Solo como respaldo para poder ejecutar el notebook sin internet: datos SINTÉTICOS.
    import random
    random.seed(42)
    simbolos = ["AAPL","MSFT","GOOG","AMZN","TSLA","META","NFLX","NVDA","IBM","ORCL"]
    with open("DATOS/stocks.txt", "w") as f:
        for s in simbolos:
            p = random.uniform(50, 400)
            for d in range(100):
                p *= random.uniform(0.97, 1.03)
                f.write(f"{s},2020-{1 + d // 28:02d}-{1 + d % 28:02d},{p:.2f},{random.randint(10**6, 10**8)}\n")
    print("⚠️ No hubo internet: se generó un stocks.txt SINTÉTICO. Los resultados no son los del dataset real.")

# 4. Ruta absoluta con file://
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

# 5. Cargar
rdd = sc.textFile(path_final)

# 6. Verificar
print(rdd.take(5))

['id,name,category,quantity,price', '1,iPhone,Electronics,10,899.99', '2,Macbook,Electronics,5,1299.99', '3,iPad,Electronics,15,499.99', '4,Samsung TV,Electronics,8,799.99']


## Paso 2: Limpieza y parseo de datos

In [11]:
# Detectar el formato mirando el encabezado
header = rdd.first().replace(" ", "").lower()
if header.startswith("id,name,category"):
    MODO = "inventario"        # id,name,category,quantity,price  (archivo real)
    ETQ_G, ETQ_K, UNID = "categoría", "producto", "unidades"
else:
    MODO = "bolsa"             # Símbolo,Fecha,Precio,Volumen     (enunciado)
    ETQ_G, ETQ_K, UNID = "símbolo", "día", "acciones"
print("Formato detectado:", MODO)

def parsear_linea(linea):
    # Devuelve (grupo, etiqueta, precio, cantidad) o None si la línea no es válida
    try:
        p = [x.strip() for x in linea.split(",")]     # un único split por línea
        if MODO == "inventario":
            return (p[2], p[1], float(p[4]), int(float(p[3])))   # (category, name, price, quantity)
        return (p[0], p[1], float(p[2]), int(float(p[3])))       # (símbolo, fecha, precio, volumen)
    except (ValueError, IndexError):
        return None                                  # encabezados, líneas vacías o mal formadas

# Aplicar la función y filtrar nulos (se cachea porque se reutiliza en todos los pasos siguientes)
rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None).cache()

# Contar líneas válidas
print(f"Líneas leídas   : {rdd.count():,}")
print(f"Líneas válidas  : {rdd_parseado.count():,}")
if rdd_parseado.count() == 0:
    print("⚠️ Ninguna línea se pudo parsear. Revisa el formato en las primeras líneas impresas arriba (¿separador distinto de coma?).")

Formato detectado: inventario
Líneas leídas   : 31
Líneas válidas  : 30


## Paso 3: Análisis por símbolo – estadísticas básicas

In [12]:
# 1. Pares (símbolo, precio)
simbolo_precio = rdd_parseado.map(lambda x: (x[0], x[2]))

# 2. Máximo, mínimo y promedio por símbolo
precio_max = simbolo_precio.reduceByKey(max)
precio_min = simbolo_precio.reduceByKey(min)

precio_promedio = simbolo_precio.aggregateByKey(
    (0.0, 0),                                          # (suma, contador)
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),   # combinar valor con acumulador (dentro de la partición)
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])   # combinar acumuladores de distintas particiones
).mapValues(lambda x: x[0] / x[1])

# 3. Resultados ordenados por símbolo
print(f"Precio máximo por {ETQ_G}:")
for simbolo, precio in sorted(precio_max.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

print(f"\nPrecio mínimo por {ETQ_G}:")
for simbolo, precio in sorted(precio_min.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

print(f"\nPrecio promedio por {ETQ_G}:")
for simbolo, precio in sorted(precio_promedio.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

Precio máximo por categoría:
  Accessories: $29.99
  Clothing: $199.99
  Electronics: $1299.99
  Food: $5.99
  Furniture: $249.99
  Sports: $49.99

Precio mínimo por categoría:
  Accessories: $24.99
  Clothing: $14.99
  Electronics: $129.99
  Food: $0.25
  Furniture: $79.99
  Sports: $19.99

Precio promedio por categoría:
  Accessories: $27.49
  Clothing: $99.28
  Electronics: $586.66
  Food: $2.30
  Furniture: $141.99
  Sports: $34.99


## Paso 4: Análisis de volumen – top días de mayor actividad

In [13]:
# 1. Pares (etiqueta, cantidad): fecha→volumen en bolsa, producto→unidades en inventario
fecha_volumen = rdd_parseado.map(lambda x: (x[1], x[3]))

# 2. Sumar volumen por fecha
volumen_por_fecha = fecha_volumen.reduceByKey(lambda a, b: a + b)

# 3. Ordenar descendente por volumen
top_fechas = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False)

# 4. Top 10
print(f"Top 10 ({ETQ_K}) con mayor cantidad:")
for fecha, volumen in top_fechas.take(10):
    print(f"  {fecha}: {volumen:,} {UNID}")

Top 10 (producto) con mayor cantidad:
  Bananas: 150 unidades
  Oranges: 120 unidades
  Apples: 100 unidades
  Chicken Breast: 50 unidades
  T-shirt: 50 unidades
  Sneakers: 40 unidades
  Backpack: 30 unidades
  Nike Shoes: 30 unidades
  Salmon Fillet: 30 unidades
  Jeans: 30 unidades


## Paso 5: Análisis avanzado – volatilidad por símbolo

Volatilidad = `(precio_max - precio_min) / precio_min * 100`. Se calcula máximo y mínimo **en una sola pasada** con `aggregateByKey`.

In [14]:
# 1. Max y min por símbolo en una sola pasada
stats_por_simbolo = simbolo_precio.aggregateByKey(
    (float('-inf'), float('inf')),                                     # (max, min)
    lambda acc, precio: (max(acc[0], precio), min(acc[1], precio)),    # actualizar max y min
    lambda acc1, acc2: (max(acc1[0], acc2[0]), min(acc1[1], acc2[1]))  # combinar acumuladores
)

# 2. Volatilidad (si el mínimo fuera 0 se evita dividir entre cero)
volatilidad = stats_por_simbolo.mapValues(
    lambda x: ((x[0] - x[1]) / x[1]) * 100 if x[1] > 0 else float('nan')
).filter(lambda kv: kv[1] == kv[1])      # descarta NaN

# 3 y 4. Más volátiles y más estables
mas_volatiles = volatilidad.sortBy(lambda x: x[1], ascending=False).take(5)
mas_estables  = volatilidad.sortBy(lambda x: x[1], ascending=True).take(5)

print(f"{ETQ_G.capitalize()}s más volátiles:")
for simbolo, vol in mas_volatiles:
    print(f"  {simbolo}: {vol:.2f}%")

print(f"\n{ETQ_G.capitalize()}s más estables:")
for simbolo, vol in mas_estables:
    print(f"  {simbolo}: {vol:.2f}%")

Categorías más volátiles:
  Food: 2296.00%
  Clothing: 1234.16%
  Electronics: 900.07%
  Furniture: 212.53%
  Sports: 150.08%

Categorías más estables:
  Accessories: 20.01%
  Sports: 150.08%
  Furniture: 212.53%
  Electronics: 900.07%
  Clothing: 1234.16%


## Paso 6: Guardar resultados y análisis final

In [15]:
import shutil

def borrar_si_existe(ruta_relativa):
    # saveAsTextFile falla si la carpeta de salida ya existe
    shutil.rmtree(ruta_relativa, ignore_errors=True)

# 1 y 2. Guardar resultados en archivos de texto (carpeta RESULTADOS creada en la preparación)
borrar_si_existe("./RESULTADOS/volumen_por_fecha")
output_path_volumen = "file://" + os.path.abspath("./RESULTADOS/volumen_por_fecha")
volumen_por_fecha.saveAsTextFile(output_path_volumen)

borrar_si_existe("./RESULTADOS/stats_por_simbolo")
output_path_stats = "file://" + os.path.abspath("./RESULTADOS/stats_por_simbolo")
stats_por_simbolo.saveAsTextFile(output_path_stats)

# 3. Métricas para el resumen ejecutivo
total_registros      = rdd_parseado.count()
simbolos_unicos      = rdd_parseado.map(lambda x: x[0]).distinct().count()
simbolo_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
dia_mayor_volumen    = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False).first()
simbolo_mas_volatil  = volatilidad.sortBy(lambda x: x[1], ascending=False).first()

print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS")
print("=" * 60)
print(f"\nTotal de registros procesados: {total_registros:,}")
print(f"Número de {ETQ_G}s distintos: {simbolos_unicos}")
print(f"\n{ETQ_G.capitalize()} con mayor precio promedio: {simbolo_mayor_precio[0]} (${simbolo_mayor_precio[1]:.2f})")
print(f"{ETQ_K.capitalize()} con mayor cantidad: {dia_mayor_volumen[0]} ({dia_mayor_volumen[1]:,} {UNID})")
print(f"{ETQ_G.capitalize()} más volátil: {simbolo_mas_volatil[0]} ({simbolo_mas_volatil[1]:.2f}%)")

# 4. Cerrar sesión
print("\nArchivos guardados en ./RESULTADOS:", os.listdir("RESULTADOS"))
spark.stop()
print("\n✅ Análisis completado. Sesión cerrada.")


RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS

Total de registros procesados: 30
Número de categorías distintos: 6

Categoría con mayor precio promedio: Electronics ($586.66)
Producto con mayor cantidad: Bananas (150 unidades)
Categoría más volátil: Food (2296.00%)

Archivos guardados en ./RESULTADOS: ['volumen_por_fecha', 'stats_por_simbolo']

✅ Análisis completado. Sesión cerrada.


---
# Preguntas de reflexión

In [16]:
# Demostración para las preguntas 1 y 2 (se ejecuta en Python puro y en Spark local)
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Reflexion").getOrCreate()
sc = spark.sparkContext

# Pregunta 1: coste de hacer split 3 veces vs 1 vez sobre 1 millón de líneas
lineas = [f"Persona{i},{20 + i % 60},Ciudad{i % 50}" for i in range(1_000_000)]

t0 = time.time()
for l in lineas:
    a = l.split(",")[0]; b = l.split(",")[1]; c = l.split(",")[2]
t_tres = time.time() - t0

t0 = time.time()
for l in lineas:
    a, b, c = l.split(",")
t_uno = time.time() - t0
print(f"3 splits/línea: {t_tres:.2f}s | 1 split/línea: {t_uno:.2f}s | ahorro ≈ {t_tres - t_uno:.2f}s ({(1 - t_uno / t_tres) * 100:.0f}%)")

# Pregunta 2: particiones de un RDD
rdd_demo = sc.parallelize(range(1_000_000))
print("Particiones por defecto (parallelize):", rdd_demo.getNumPartitions())
print("Particiones de textFile (stocks.txt)  :", sc.textFile("file://" + os.path.abspath("DATOS/stocks.txt")).getNumPartitions())
print("defaultParallelism:", sc.defaultParallelism)
spark.stop()

3 splits/línea: 0.80s | 1 split/línea: 0.35s | ahorro ≈ 0.46s (57%)
Particiones por defecto (parallelize): 2
Particiones de textFile (stocks.txt)  : 2
defaultParallelism: 2


**1. Eficiencia – ¿por qué es importante hacer `split()` una sola vez por línea? ¿Cuánto tiempo ahorrarías con 1 millón de líneas?**
Porque `split()` crea una lista nueva de strings cada vez que se llama. Si se llama 3 veces sobre la misma línea, se repite el mismo trabajo de CPU y de memoria. En Spark ese coste se multiplica por cada registro y se paga en todos los executors. Con 1 millón de líneas se pasa de 3 millones de splits a 1 millón (≈ 66 % menos de parseo). En mi medición (salida de la celda anterior), hacer 3 splits por línea tardó aproximadamente **el doble** que hacer 1 solo: un ahorro de alrededor del **50 %** del tiempo de parseo (entre 0.4 s y 0.8 s por millón de líneas según la corrida). Con 100 veces más datos, el ahorro sería de decenas de segundos. Además, en un job real se evita releer el archivo en cada transformación.

**2. Particionamiento – ¿cuántas particiones tiene tu RDD? ¿Cómo afecta al rendimiento? ¿Qué pasaría con solo 1?**
Mi RDD tiene **2 particiones**, tanto con `parallelize` como con `textFile` sobre `stocks.txt`. Cada partición se convierte en una tarea que procesa un core, así que más particiones permiten más paralelismo, hasta saturar los cores disponibles. Con solo **1 partición** todo se ejecutaría en una única tarea y un único core, se perdería el paralelismo y esa partición tendría que caber en la memoria de un executor. Con demasiadas particiones muy pequeñas tampoco conviene, porque el coste de planificar las tareas supera el trabajo útil. Como regla práctica se usan 2–4 particiones por core.

**3. Lazy evaluation – ¿cuándo se ejecutaron realmente las transformaciones? ¿Qué ventajas tiene?**
Las transformaciones (`map`, `filter`, `reduceByKey`, `sortBy`…) solo construyen el plan (el DAG/linaje). Se ejecutan realmente cuando se llama a una **acción**: `count()`, `take()`, `collect()`, `sum()`, `first()`, `saveAsTextFile()`, etc. (por ejemplo, en el Ejercicio 3 nada se calcula hasta `mayores50.count()`). Ventajas: Spark puede ver todo el pipeline y **optimizarlo** (encadenar `map`+`filter` en un solo *stage* sin materializar resultados intermedios), no calcula datos que nunca se usan, procesa solo lo necesario (p. ej. `take(5)` no lee el archivo completo) y guarda el linaje para **recuperar** particiones perdidas recalculándolas.

**4. Memoria – ¿qué pasaría si hicieras `collect()` en un RDD de 10 GB? ¿Cómo lo evitarías?**
`collect()` trae todos los datos de todos los executors al **driver**, que normalmente tiene 1–2 GB de memoria (`spark.driver.memory`). Con 10 GB (y más en memoria por la sobrecarga de objetos Python) el driver se queda sin memoria: `OutOfMemoryError` o `Py4JJavaError`, y la aplicación cae. Se evita con: `take(n)` / `takeSample` para una muestra, `count()`, `reduce`, `aggregate` u otras acciones que devuelven un valor pequeño, agregar antes (`reduceByKey`) para reducir el volumen, `foreach`/`foreachPartition` para procesar en los executors, o `saveAsTextFile()` para escribir el resultado a disco/HDFS sin pasar por el driver.

**5. Optimización – con 100× más datos, ¿qué cambiarías en tu código? ¿Más particiones? ¿Cachearías algún RDD?**
- **Particiones:** sí, aumentar el paralelismo (`sc.textFile(path, minPartitions=N)` o `repartition`) hasta unas 2–4 por core del cluster, y subir `spark.sql.shuffle.partitions` (ahora de 8–10 a ~200, el valor de producción).
- **Caché:** sí, `rdd_parseado` se reutiliza en todos los pasos (estadísticas, volumen, volatilidad, resumen), así que `cache()`/`persist(MEMORY_AND_DISK)` evita releer y reparsear el archivo en cada acción. No tiene sentido cachear RDDs que se usan una sola vez.
- **Una sola pasada:** calcular máx/mín/suma/contador en un único `aggregateByKey` en lugar de varias agregaciones separadas; usar `reduceByKey`/`aggregateByKey` (combinan en origen antes del shuffle) en vez de `groupByKey`.
- **Evitar `collect()`** de resultados grandes; leer desde HDFS en lugar de local; y, para datos estructurados, pasar a **DataFrames** (optimizador Catalyst, formato columnar Parquet).
- Recursos: más memoria por executor y revisar la pestaña *Stages* de la Spark UI para detectar *data skew* o cuellos de botella.

**6. HDFS vs local – ¿cuáles son las ventajas de usar HDFS en lugar de archivos locales para este análisis?**
- **Distribución y localidad de datos:** el archivo se divide en bloques repartidos por los nodos y Spark ejecuta cada tarea cerca del bloque que lee, en vez de que todos los nodos necesiten el archivo en la misma ruta local (con `file://` el archivo debe existir idéntico en cada worker).
- **Tolerancia a fallos:** los bloques están replicados (normalmente ×3); si un nodo cae, Spark lee otra réplica automáticamente.
- **Escalabilidad:** soporta archivos y conjuntos de datos mucho mayores que el disco de una sola máquina, y la lectura es paralela desde muchos nodos.
- **Simplicidad de rutas y resultados:** una única vista del sistema de archivos accesible por todo el cluster; `saveAsTextFile` escribe de forma distribuida y los resultados quedan disponibles para otros trabajos.
- **Integración** con el ecosistema Hadoop (YARN, Hive, etc.).
Para ~1000 registros no hay diferencia de rendimiento apreciable, pero HDFS es el que escala a cientos de GB/TB.